In [ ]:
## 必要パッケージ
!pip install scikit-learn

In [ ]:
## パイプライン用意
from transformers import pipeline
pipe = pipeline("text-classification", model="transformersbook/bert-base-uncased-finetuned-clinc", trust_remote_code=True)

In [ ]:
# ベンチマーク用クラス
class PerformanceBenchmark:
    def __init__(self, pipeline, dataset, optim_type="BERT baseline"):
        self.pipeline = pipeline
        self.dataset = dataset
        self.optim_type = optim_type

    def compute_accuracy(self):
        # あとで定義
        pass

    def compute_size(self):
        # あとで定義
        pass

    def time_pipeline(self):
        # あとで定義
        pass

    def run_benchmark(self):
        metrics = {}
        metrics[self.optim_type] = self.compute_size()
        metrics[self.optim_type].update(self.time_pipeline())
        metrics[self.optim_type].update(self.compute_accuracy())
        return metrics

In [ ]:
# 評価用にデータセットをロード
from datasets import load_dataset

clinc = load_dataset("clinc_oos", "plus") # スコープ外の学習事例を含む

In [ ]:
# 事例を確認
sample = clinc["test"][42]
sample # text(クエリ) と intent(意図(クラスラベル)) からなる

In [ ]:
# 意図のクラス名確認
intents = clinc["test"].features["intent"]
intents.int2str(sample["intent"]) # transfer

In [ ]:
# 指標の作成(今回は正解率)
from sklearn.metrics import accuracy_score

In [ ]:
# 正解率計算メソッド
def compute_accuracy(self):
    """PerformanceBenchmark.compute_accuracy()メソッドをオーバーライド"""
    preds, labels = [], []
    for example in self.dataset:
        pred = self.pipeline(example["text"])[0]["label"]
        label = example["intent"]
        preds.append(intents.str2int(pred))
        labels.append(label)

    accuracy = accuracy_score(labels, preds)
    print(f"Accuracy on test set - {accuracy:.3f}")
    return {"accuracy": accuracy}

PerformanceBenchmark.compute_accuracy = compute_accuracy

In [ ]:
# モデルの state_dict を確認
list(pipe.model.state_dict().items())[42] # 重みの名前と値が確認できる

In [ ]:
# # モデルの保存例
# import torch

# torch.save(pipe.model.state_dict(), "model.pt")

In [ ]:
# モデルサイズに関する情報取得メソッド
import torch
from pathlib import Path

def compute_size(self):
    """PerformanceBenchmark.compute_size()メソッドをオーバーライド"""
    # モデルの重みを一時的に保存
    temp_path = Path("model.pt")
    torch.save(self.pipeline.model.state_dict(), temp_path)

    # ファイルサイズ(MB)を計算
    size_mb = temp_path.stat().st_size / (1024 * 1024)
    print(f"Model size - {size_mb:.2f} MB")

    # 一時ファイルを削除
    temp_path.unlink()

    return {"size_mb": size_mb}

PerformanceBenchmark.compute_size = compute_size

In [ ]:
# クエリに対するパイプラインのレイテンシ確認
from time import perf_counter

## クエリ
query = """Hey, I'd like to rent a vehicle from Nov 1st to Nov 15th in
Paris and I need a 15 passenger van"""

for _ in range(3):
    start_time = perf_counter()
    _ = pipe(query)
    latency = perf_counter() - start_time
    print(f"Latency (ms) - {1000 * latency:.3f}")

In [ ]:
# パイプラインのレイテンシを計測するメソッド
import numpy as np

def time_pipeline(self, query="What is the pin number for my account?"):
    """PerformanceBenchmark.time_pipeline()メソッドをオーバーライド"""
    latencies = []
    # ウォームアップ
    for _ in range(10):
        _ = self.pipeline(query)
    # 時間の計測
    for _ in range(100):
        start_time = perf_counter()
        _ = self.pipeline(query)
        latency = perf_counter() - start_time
        latencies.append(latency)
    # 統計量計算
    time_avg_ms = 1000 * np.mean(latencies)
    time_std_ms = 1000 * np.std(latencies)
    print(f"Average latency (ms) - {time_avg_ms:.2f} +\- {time_std_ms:.2f}")
    return {"time_avg_ms": time_avg_ms, "time_std_ms": time_std_ms}

PerformanceBenchmark.time_pipeline = time_pipeline

In [ ]:
# BERT のベンチマークを確認
pb = PerformanceBenchmark(pipe, clinc["test"])
perf_metrics = pb.run_benchmark()